# 02 — Feature EDA (Phase 2)

One row per H3 hex, ~50–100 features describing built form, POIs, land use,
network, transit, population, centrality and terrain (README §7 Phase 2).
This notebook checks the feature tables before they feed the Phase 3 job model:

1. **Distributions** — are the heavy-tailed counts tamed by the configured log1p?
2. **Correlations** — what near-duplicates were dropped, what structure remains?
3. **Maps** — do GFA / POIs / centrality match what we know about Almaty
   (strong centre around Almaly/Medeu, industrial north along the railway)?
4. **Moran's I** — are the key features spatially clustered (they should be)?

By default this runs on the **committed sample** (offline, seconds). For the
full city, set the environment variable first:
`ALMATY_CONFIG=config/config.yaml` (needs `almaty-access prepare` to have run).

In [ ]:
import json
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from almaty_access.config import load_config
from almaty_access.features.build import REPORT_FILE, build_features
from almaty_access.schemas import FEATURE_ID_COLUMNS

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
CONFIG_PATH = Path(os.environ.get("ALMATY_CONFIG", REPO / "config" / "config.sample.yaml"))
config = load_config(CONFIG_PATH)
build_features(config)  # cached: rebuilds only when inputs changed

P = config.paths.processed
grid8 = gpd.read_parquet(P / "grid_r8.parquet")
grid9 = gpd.read_parquet(P / "grid_r9.parquet")
feat8 = pd.read_parquet(P / "features_r8.parquet")
feat9 = pd.read_parquet(P / "features_r9.parquet")
report = json.loads((P / REPORT_FILE).read_text("utf-8"))
FEATURES = [c for c in feat8.columns if c not in FEATURE_ID_COLUMNS]
METRIC = config.crs.metric
print(f"config: {CONFIG_PATH.name}")
print(f"r8: {len(feat8):,} cells, r9: {len(feat9):,} cells, {len(FEATURES)} features")
print(
    f"near-duplicates dropped: {len(report['near_duplicates_dropped'])}"
    + (
        f" (pruning skipped: {report['pruning_skipped_reason']})"
        if report["pruning_skipped_reason"]
        else ""
    )
)
print(f"sub-centres found: {report['n_subcenters']}")

## What was dropped and why

The build prunes one of every feature pair with |r| > threshold on the
**log-transformed r8 table** (raw heavy-tailed counts make Pearson r
meaningless) — except that README-named core features are **protected**:
they are never dropped, and a high-r pair of protected columns is recorded
under `high_corr_kept` instead. Expected regular drops: densities (an exact
multiple of the count on a fixed-area grid), k-ring means vs sums (disk
sizes are nearly constant), and `tr_stop_cnt` (the same physical stops that
`poi_cnt_transit_stop` already counts).

In [ ]:
if report["high_corr_kept"]:
    print("high-|r| pairs kept because both columns are protected:")
    for pair in report["high_corr_kept"]:
        print(f"  {pair['a']}  ~  {pair['b']}  (r={pair['r']})")
pd.DataFrame(report["near_duplicates_dropped"])

## Distributions

Counts and areas are heavy-tailed (most hexes near zero, a few huge).
The configured `log1p` turns them into something a model can use;
`features_report.json` lists exactly which columns get it.

In [ ]:
show = [
    c
    for c in [
        "bf_gfa_m2",
        "bf_gfa_residential_m2",
        "bf_gfa_commercial_m2",
        "bf_bldg_cnt",
        "poi_cnt_total",
        "pop_residents",
        "net_len_drive_m",
        "tr_stop_cnt",
    ]
    if c in feat9.columns
][:8]
log_cols = set(report["log1p_columns"])
fig, axes = plt.subplots(2, max(len(show) // 2, 1), figsize=(16, 6))
for ax, col in zip(axes.ravel(), show, strict=False):
    raw = feat9[col]
    values = np.log1p(raw) if col in log_cols else raw
    ax.hist(values, bins=40, color="#30808f")
    ax.set_title(("log1p " if col in log_cols else "") + col, fontsize=9)
fig.suptitle("Feature distributions at r9 (transformed as the model will see them)")
fig.tight_layout()

## Correlation structure (after pruning)

Blocks of related features should correlate moderately (built form with POIs,
both with population); anything still ≈ ±1 would mean the pruning threshold
or the feature design needs a second look.

In [ ]:
transformed = feat8[FEATURES].copy()
for col in report["log1p_columns"]:
    if col in transformed.columns:
        transformed[col] = np.log1p(transformed[col])
corr = transformed.corr()
fig, ax = plt.subplots(figsize=(14, 12))
image = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=6)
ax.set_yticks(range(len(corr)), corr.columns, fontsize=6)
fig.colorbar(image, shrink=0.6, label="Pearson r (log1p view)")
ax.set_title(f"Correlations, r8, {len(FEATURES)} features after pruning")
fig.tight_layout()
remaining = corr.abs().where(~np.eye(len(corr), dtype=bool)).max().max()
print(f"largest remaining |r|: {remaining:.3f} (threshold {report['corr_threshold']})")

## Maps of the top features

The README's "done when": GFA and POI maps must visually match Almaty —
a strong centre around Almaly / lower Medeu, the industrial belt along the
northern railway, little mass in the mountain south. On the sample
(a small central bbox) expect a dense, fairly uniform urban core instead.

In [ ]:
joined = grid9.merge(feat9, on="h3", suffixes=("", "_f")).to_crs(METRIC)
panels = [
    c
    for c in [
        "bf_gfa_m2",
        "bf_gfa_commercial_m2",
        "poi_cnt_total",
        "poi_entropy",
        "net_betweenness_max",
        "cent_dist_center_m",
    ]
    if c in feat9.columns
][:6]
log_cols = set(report["log1p_columns"])
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, col in zip(axes.ravel(), panels, strict=False):
    values = np.log1p(joined[col]) if col in log_cols else joined[col]
    joined.assign(v=values).plot(
        ax=ax, column="v", cmap="magma", legend=True, legend_kwds={"shrink": 0.6}
    )
    ax.set_title(("log1p " if col in log_cols else "") + col, fontsize=10)
    ax.set_axis_off()
fig.suptitle("Key features at r9")
fig.tight_layout()

## Moran's I — is the structure spatial?

Moran's I measures whether similar values neighbour each other
(+1 clustered, 0 random). Urban structure *is* spatial, so the key features
should show clearly positive, significant values; if they did not, the
features (or the grid join) would be broken. Implementation:
`almaty_access.features.spatial` (H3 ring-1 neighbours, permutation test).

In [ ]:
from almaty_access.features.build import MIN_CELLS_FOR_PRUNING
from almaty_access.features.spatial import morans_i

# Spatial statistics need cells to be statistical about: prefer r8, fall
# back to r9 on the tiny sample grid.
basis_name, basis = ("r8", feat8) if len(feat8) >= MIN_CELLS_FOR_PRUNING else ("r9", feat9)
print(f"Moran's I computed on {basis_name} ({len(basis):,} cells)")
key = [
    c
    for c in [
        "bf_gfa_m2",
        "bf_gfa_residential_m2",
        "bf_gfa_commercial_m2",
        "poi_cnt_total",
        "pop_residents",
        "net_betweenness_max",
        "cent_dist_center_m",
        "ter_slope_mean_deg",
    ]
    if c in basis.columns
]
indexed = basis.set_index("h3")
rows = []
for col in key:
    result = morans_i(indexed[col], n_perm=199, seed=config.random_seed)
    rows.append({"feature": col, "morans_i": round(result.i, 3), "p_sim": round(result.p_sim, 4)})
moran_table = pd.DataFrame(rows)
moran_table

## Verdict

In [ ]:
checks = {
    "identical schema r8/r9": list(feat8.columns) == list(feat9.columns),
    "no NaN/inf in features": bool(
        np.isfinite(feat9[FEATURES].to_numpy(float)).all()
        and np.isfinite(feat8[FEATURES].to_numpy(float)).all()
    ),
    "feature count in 50-100": 50 <= len(FEATURES) <= 100,
    "every city hex has a district": bool(grid9.loc[grid9["in_city"], "district"].notna().all()),
    "built mass present": bool(feat9["bf_gfa_residential_m2"].sum() > 0),
    # A full-city criterion: on the tiny sample grid Moran's I has too few
    # cells to mean anything, so the check is skipped there (None).
    "key features spatially clustered": (
        bool((moran_table["morans_i"] > 0.2).mean() >= 0.5)
        if len(basis) >= MIN_CELLS_FOR_PRUNING
        else None
    ),
}
for name, ok in checks.items():
    print(("SKIP " if ok is None else "PASS " if ok else "FAIL ") + name)
assert all(ok for ok in checks.values() if ok is not None), "Phase 2 'done when' not satisfied"
print("\nPhase 2 feature tables look sound; ready for the Phase 3 job model.")